In [ ]:
%%capture
pip install chromadb openai

Chroma can store documents with optional metadata, and when you query it, you can return the matching text plus metadata like filename and chunk number.

OpenRouter is OpenAI-compatible, so we can keep using the openai Python SDK with the OpenRouter base URL.

In [ ]:
import os
import uuid
import chromadb
from openai import OpenAI

In [ ]:
from google.colab import userdata
api_key=userdata.get('main_api')
llm = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)


In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="my_rag_collection"
)


In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
with open('/content/news_text.text', "r", encoding="utf-8") as file:
        text = file.read()

In [ ]:
def index_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()

    chunks = chunk_text(text)

    documents = []
    ids = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        documents.append(chunk)
        ids.append(str(uuid.uuid4()))
        metadatas.append({
            "source": file_path,
            "chunk": i
        })

    collection.add(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )

    print(f"Indexed {len(chunks)} chunks from {file_path}")


In [ ]:
index_file('/content/news_text.text')

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s] 


Indexed 46 chunks from /content/news_text.text


In [ ]:
def retrieve_context(question, n_results=2):
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    context_blocks = []

    for doc, meta in zip(documents, metadatas):
        source = meta.get("source", "unknown")
        chunk = meta.get("chunk", "unknown")

        context_blocks.append(
            f"Source: {source}, Chunk: {chunk}\n{doc}"
        )

    return "\n\n---\n\n".join(context_blocks)

In [ ]:
retrieve_context('what is new thing on sport')

"Source: /content/news_text.text, Chunk: 36\nwould demand £120m\nMissed putts & hot yoga - Hirst on thrill of Scotland call\nAll you need to know about the 2026 BBC Football Awards\nHealth\nThree young girls playing in circle and pressing their hands together in a school sports hall (Credit: Getty Images)\nChildren need to move more. Here's how to help\nChildren are less physically active than they used to be. Scientists are finding effective ways to encourage children to move more, leaving lasting benefits on their health.\n\nWhy do we want to b\n\n---\n\nSource: /content/news_text.text, Chunk: 2\nan court has ruled that a hotel acted lawfully when it refused to provide tap water to a tourist. Additionally, Germany's most wanted woman has been jailed after evading police for three decades.SportsFootball: The US Men's National Team (USMNT) World Cup squad reveal has prompted reactions regarding key winners and losers.Basketball: Kelsey Plum has been sidelined with an injury in practice

In [ ]:
def ask_rag(question):
    context = retrieve_context(question)

    system_prompt = """
You are a helpful RAG assistant.
Answer using only the provided context.
If the answer is not in the context, say:
"I don't know based on the provided documents."

At the end, include the source names you used.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}
"""

    response = llm.chat.completions.create(
        model="qwen/qwen3.6-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=300,
        temperature=0,
    )

    return response.choices[0].message.content

In [ ]:
#index_file("/content/news_text.text")
while True:
  question = input("\nAsk a question, or type 'exit': ")
  if question.lower() == "exit":
    break

  answer = ask_rag(question)
  print("\nAnswer:")
  print(answer)

Indexed 46 chunks from /content/news_text.text

Ask a question, or type 'exit': what new thing on iran

Answer:
Based on the provided documents, the US and Iran are currently exploring a draft peace deal. This development has also contributed to cooling global oil prices.

Source: /content/news_text.text

Ask a question, or type 'exit': what  happened on sport today

Answer:
Based on the provided documents, here are the sports updates:
- **Football:** The US Men's National Team (USMNT) World Cup squad reveal prompted reactions regarding key winners and losers.
- **Basketball:** Kelsey Plum was sidelined with an injury in practice, and Karl-Anthony Towns (KAT) trolled the Cavaliers on social media.
- **Wrestling:** WWE star Logan Paul suffered (the provided text cuts off here).

Source: /content/news_text.text

Ask a question, or type 'exit': quit

Answer:
I don't know based on the provided documents.
Source: /content/news_text.text

Ask a question, or type 'exit': exit


In [ ]:
%%capture
!pip install pypdf

In [ ]:
import os
import chromadb
from pypdf import PdfReader
from openai import OpenAI

In [ ]:
llm = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="pdf_rag_collection"
)


In [ ]:
def chunk_text(text, chunk_size=700, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
pdf=PdfReader('/content/Yisak Bule resume narrow.pdf')

In [ ]:
pdf.pages

In [ ]:
for page_number, page in enumerate(pdf.pages, start=1):
    text = page.extract_text()
    print(f"Page {page_number}:\n{text}\n")

Page 1:
  Yisak  Bule                        Addis  Ababa,  Ethiopia                       +251934841558                                              Gmail    linkedin  Zindi kaggle    Github                                                                                                                      Data  Scientist|machine  learning  engineer I  am  a  results-driven  data  scientist  and  machine  learning  engineer  with  a  strong  desire  to  solve  real-world  
challenges.
 Highly  accomplished  Machine  Learning  Engineer  with  a  proven  track  record  of  building  
high-accuracy
 
models
 
in
 
diverse
 
industries
 
like
 
healthcare,
 
telecom,
 
agriculture,
 
and
 
business.
 
Recognized
 
for
 
winning
 
multiple
 
data
 
science
 
competitions,
 
demonstrating
 
exceptional
 
problem-solving
 
and
 
technical
 
skills.
 
Eager
 
to
 
leverage
 
expertise
 
in
 
Python,
 
Scikit-learn,
 
TensorFlow,
 
and
 
other
 
libraries
 
to
 
contribute
 
impactful
 
soluti

In [ ]:
#extract_pdf_pages('/content/Yisak Bule resume narrow.pdf')

In [ ]:
# Extract PDF page text
# -----------------------------
def extract_pdf_pages(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            pages.append({
                "page": page_number,
                "text": text.strip()
            })

    return pages

In [ ]:
pages=extract_pdf_pages('/content/Yisak Bule resume narrow.pdf')

In [ ]:
pages[2]['page']

3

In [ ]:
chunk_text(pages[0]['text'])[1]

'track  record  of  building  \nhigh-accuracy\n \nmodels\n \nin\n \ndiverse\n \nindustries\n \nlike\n \nhealthcare,\n \ntelecom,\n \nagriculture,\n \nand\n \nbusiness.\n \nRecognized\n \nfor\n \nwinning\n \nmultiple\n \ndata\n \nscience\n \ncompetitions,\n \ndemonstrating\n \nexceptional\n \nproblem-solving\n \nand\n \ntechnical\n \nskills.\n \nEager\n \nto\n \nleverage\n \nexpertise\n \nin\n \nPython,\n \nScikit-learn,\n \nTensorFlow,\n \nand\n \nother\n \nlibraries\n \nto\n \ncontribute\n \nimpactful\n \nsolutions\n \nand\n \ncontinuously\n \nlearn\n \nwithin\n \na\n \ncollaborative\n \nteam. \n \nWork  Experience Data  Scientist|machine  learning  engineer(2023-2024)  Kaggle  -  Predict  Energy  Behavior  of  Prosumers  Competition:  Bronze  Medalist  (Ranked  within  Top  \n10%)'

In [ ]:
def index_pdf(pdf_path):
    pages = extract_pdf_pages(pdf_path)

    documents = []
    ids = []
    metadatas = []

    for page in pages:
        page_number = page["page"]
        page_text = page["text"]

        chunks = chunk_text(page_text)

        for chunk_index, chunk in enumerate(chunks):
            doc_id = f"{pdf_path}_page_{page_number}_chunk_{chunk_index}"

            documents.append(chunk)
            ids.append(doc_id)
            metadatas.append({
                "source": pdf_path,
                "page": page_number,
                "chunk": chunk_index
            })

    collection.upsert(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )

    print(f"Indexed {len(documents)} chunks from {pdf_path}")



In [ ]:
index_pdf('/content/Yisak Bule resume narrow.pdf')

Indexed 13 chunks from /content/Yisak Bule resume narrow.pdf


In [ ]:
# -----------------------------
# Retrieve relevant PDF chunks
# -----------------------------
def retrieve_context(question, n_results=3):
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    context_blocks = []

    for doc, meta in zip(documents, metadatas):
        source = meta.get("source", "unknown")
        page = meta.get("page", "unknown")
        chunk = meta.get("chunk", "unknown")

        context_blocks.append(
            f"[Source: {source}, Page: {page}, Chunk: {chunk}]\n{doc}"
        )

    return "\n\n---\n\n".join(context_blocks)

In [ ]:
retrieve_context('what are main skills')

"[Source: /content/Yisak Bule resume narrow.pdf, Page: 2, Chunk: 2]\ninitiatives,  \ncommunication\n \nplatform\n \nimprovements,\n \nand\n \nlocation-specific\n \nstrategies.\n \n Skills:  ●  Programming  Languages:  Python  (expert),  SQL  (proficient)  ●  Machine  Learning:  Linear  Regression,  Logistic  Regression,  Decision  Trees,  Random  Forests,  \nSupport\n \nVector\n \nMachines,\n \nBoosting\n \nTrees,\n \nK-means,\n \nPCA,\n \nRF\n \nmodels,\n \nXGBoost,\n \nProphet\n \n(expert)\n ●  Deep  Learning:  NLP  (intermediate),  Computer  Vision  (expert)  ●  Data  Visualization:  Tableau  (proficient),  Matplotlib,  Seaborn  (intermediate)  ●  Data  Cleaning:  Pandas,  NumPy  (expert)  ●  Statistical  Modeling:  Hypothesis  Testing,  ANOVA,  T-Tests  (proficient)  ●  Cloud  Computing:  AW\n\n---\n\n[Source: /content/Yisak Bule resume narrow.pdf, Page: 1, Chunk: 6]\nry,  grade-specific  quizzes,  and  personalized  \nscorecards,\n \nboosting\n \nengagement\n \nand\n \npromoting\n

In [ ]:
# -----------------------------
# Ask Qwen with PDF context
# -----------------------------
def ask_pdf_rag(question):
    context = retrieve_context(question)

    system_prompt = """
You are a helpful PDF RAG assistant.

Rules:
1. Answer using only the provided context.
2. If the answer is not in the context, say:
   "I don't know based on the provided PDF."
3. Include citations using this format:
   (Source: filename, Page: page number)
4. Be clear and concise.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}
"""

    response = llm.chat.completions.create(
        model="qwen/qwen3.6-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=300,
        temperature=0,
    )

    return response.choices[0].message.content


In [ ]:
pdf_path = "/content/Yisak Bule resume narrow.pdf"
index_pdf(pdf_path)


Indexed 13 chunks from /content/Yisak Bule resume narrow.pdf


In [ ]:
while True:
  question = input("\nAsk a question about the PDF, or type 'exit': ")
  if question.lower() == "exit":
    break

  answer = ask_pdf_rag(question)
  print("\nAnswer:")
  print(answer)


Ask a question about the PDF, or type 'exit': what are main skills 

Answer:
Based on the provided resume, the main skills are:

- **Programming Languages:** Python (expert), SQL (proficient)
- **Machine Learning:** Linear Regression, Logistic Regression, Decision Trees, Random Forests, Support Vector Machines, Boosting Trees, K-means, PCA, RF models, XGBoost, Prophet (expert)
- **Deep Learning:** NLP (intermediate), Computer Vision (expert)
- **Data Visualization:** Tableau (proficient), Matplotlib, Seaborn (intermediate)
- **Data Cleaning:** Pandas, NumPy (expert)
- **Statistical Modeling:** Hypothesis Testing, ANOVA, T-Tests (proficient)
- **Cloud Computing:** AWS, Azure (proficient)
- **Web Frameworks:** Django (intermediate)
- **Tools:** Scikit-learn, TensorFlow, Keras, MySQL

(Source: /content/Yisak Bule resume narrow.pdf, Page: 2)

Ask a question about the PDF, or type 'exit': exit
